# LogVar2FJ 1 — the model, and one fit of a world it owns

A two-factor log-variance model for equity index options, and what a calibration of it produces.
Everything below is a JSON document run through `derivus.Context`; the claims are the sentences
and the numbers under them are what the engine printed.

The world here is synthetic on purpose. `tests/fixtures/data/logvar2fj_world.json` carries a
hand-authored index `INDEX_A` with a five-expiry skewed ladder, sloping EUR and USD curves and a
dividend yield, so every axis a fit could confuse is varied and no market data is involved.

## The model

Two Ornstein-Uhlenbeck factors drive the log-variance of the spot. A slow one carries
`Kappa_L`, `Sigma_L`, `Rho_L`; a fast one carries `Kappa_S`, `Sigma_S`, `Rho_S`. The spot's
return over a step is Gaussian given the path of the variance.

The part of a return the leverage does not explain spends the period's **variance clock**
`A = sum_k c_k V_k`, `c = 1 - rho_s^2 - rho_l^2`, and carries a **normal-inverse-Gaussian**
residual `X_A ~ NIG(Alpha, Beta, delta_A, mu_A)` on it. `Alpha` is the tail and the smile's
convexity, `Beta` is its skew. Both `delta_A` and `mu_A` are linear in the clock, so a month cut
into days is one law once the clock is one number, and the drift is forced —
`E[exp(X_A)] = 1` — rather than fitted. There is no intensity, no jump size and no compensator,
so there is no parameter the Greeks cannot reach.

`Xi_Curve` is the forward-variance curve `xi(t) = E_0[h_t]`, the **expected** forward variance,
piecewise constant between at-the-money expiries. The OU level is derived from it,
`L*(t) = log xi(t) - Var(l_t + s_t)/2`, which makes the at-the-money level invariant to the
vol-of-vol by construction. The strip is re-bootstrapped at every outer iterate, so every
candidate reprices the at-the-money term structure exactly and is judged on the smile alone.

`Cap_A` is a hard corner on the log-variance, `min(l + s, Cap_A)`, **in simulation only**. The
fit runs unbounded and writes the level once, from the fitted stationary law, so that no path-day
reaches it on the factor that was fitted.

In [1]:
import copy, io, json, logging, os, sys, time

HERE = os.path.abspath(os.getcwd())
REPO = HERE if os.path.isdir(os.path.join(HERE, 'derivus')) else os.path.dirname(HERE)
sys.path.insert(0, REPO)

import torch
import derivus as rf
from derivus.config import CustomJsonEncoder

WORLD = os.path.join(REPO, 'tests', 'fixtures', 'data', 'logvar2fj_world.json')
MARKET = json.load(open(WORLD))['MarketData']
BASE = MARKET['System Parameters']['Base_Date']['.Timestamp']
FACTOR, BLOCK = 'LogVar2FJModelParameters.INDEX_A', 'LogVar2FJModelPrices.INDEX_A'

print('derivus   ', os.path.dirname(rf.__file__))
print('device    ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('world     ', os.path.relpath(WORLD, REPO), '- base date', BASE)

derivus    C:\Users\Vretiel\PycharmProjects\derivus\.claude\worktrees\agent-aec710af441821806\derivus
device     NVIDIA GeForce RTX 3090
world      tests\fixtures\data\logvar2fj_world.json - base date 2024-06-28


In [2]:
def job(sections, calculation=None, deals=()):
    """One document: the world file, one calculation, one book, and the sections overridden on
    top of the file. `Price Factors` and `Market Prices` merge per key."""
    calculation = calculation or {'Object': 'BaseValuation', 'Base_Date': {'.Timestamp': BASE},
                                  'Currency': 'USD', 'Greeks': 'No', 'MCMC_Simulations': 8192,
                                  'Random_Seed': 1}
    return {'Calc': {'Calculation': calculation,
                     'MergeMarketData': {'MarketDataFile': WORLD,
                                         'ExplicitMarketData': dict(sections)},
                     'Deals': {'Reference': 'notebook', 'Tag_Titles': '',
                               'Deals': {'Children': list(deals)}}}}


def fit(sections, name):
    """`(the written factor, the report the fit printed)`. Every root handler is replaced for the
    duration, so the report is this notebook's to render rather than the kernel's to spill."""
    buf, root = io.StringIO(), logging.getLogger()
    saved, level = root.handlers[:], root.level
    root.handlers, root.level = [logging.StreamHandler(buf)], logging.INFO
    try:
        cx = rf.Context()
        cx.load_json((json.dumps(job(sections), cls=CustomJsonEncoder), name + '.json'))
        cx.bootstrap()
    finally:
        root.handlers, root.level = saved, level
    return cx.current_cfg.params['Price Factors'].get(FACTOR), buf.getvalue()


def show(report, *markers, prefix=''):
    """The report's own lines, whole, in the order it wrote them."""
    for line in report.splitlines():
        if any(marker in line for marker in markers):
            print(prefix + line.strip())


def floats_of(factor):
    """Every float a written factor holds, keyed by its own path, in hex - the only comparison
    that catches a one-ulp move."""
    out = {}
    for name, value in sorted(factor.items()):
        array = getattr(value, 'array', None)
        if array is not None:
            out.update({'%s[%d][%d]' % (name, i, j): float(x).hex()
                        for i, row in enumerate(array.tolist()) for j, x in enumerate(row)})
        elif isinstance(value, (int, float)) and not isinstance(value, bool):
            out[name] = float(value).hex()
    return out

## The document the fit reads

A `LogVar2FJModelPrices` block names the factors its quotes are read against — the spot, the
volatility surface, the discount curve and the dividend yield — the quote convention, and the
solver's own settings. Its rows are the ladder.

In [3]:
block = copy.deepcopy(MARKET['Market Prices'][BLOCK])
block['instrument']['Max_Iterations'] = 60
for name, value in block['instrument'].items():
    if name != 'European_Options':
        print('%-18s %s' % (name, value))

Underlying         INDEX_A
Underlying_Type    EquityPrice
Volatility         INDEX_A.EUR
Volatility_Type    EquityPriceVol
Discount_Rate      EUR
Discount_Rate_Type InterestRate
Yield              INDEX_A
Yield_Type         DividendRate
Quote_Type         Implied_Volatility
Use_Forward        No
Invert_Moneyness   No
Steps_Per_Year     252.0
Paths              2048
Max_Iterations     60
Random_Seed        1
Sampling           Pseudo


`Quote_Type` is the convention the rows are read in; `Paths` and `Sampling` govern the forward
and simulated rows, not the vanillas, which price by quadrature. `Max_Iterations` is raised from
the fixture's 20 so that no stage stops on its budget.

In [4]:
rows = block['instrument']['European_Options']
print('%d quotes over %d expiries, quoted as %s\n' % (
    len(rows), len({r['Expiry_Date']['.Timestamp'] for r in rows}),
    block['instrument']['Quote_Type']))
print('%-12s %10s %6s %8s %8s' % ('expiry', 'strike', 'type', 'vol', 'weight'))
for row in rows:
    print('%-12s %10.4f %6s %8.5f %8.2f' % (
        row['Expiry_Date']['.Timestamp'], row['Strike'], row['Option_Type'],
        row['Quoted_Market_Value'], row['Weight']))

15 quotes over 5 expiries, quoted as Implied_Volatility

expiry           strike   type      vol   weight
2024-07-26      90.1036    Put  0.23238     1.00
2024-07-26     100.1151   Call  0.20077     1.00
2024-07-26     110.1266   Call  0.17217     1.00
2024-09-27      90.3372    Put  0.23410     1.00
2024-09-27     100.3747   Call  0.20249     1.00
2024-09-27     110.4121   Call  0.17390     1.00
2024-12-27      90.6757    Put  0.23659     1.00
2024-12-27     100.7507   Call  0.20499     1.00
2024-12-27     110.8258   Call  0.17639     1.00
2025-03-28      91.0154    Put  0.23909     1.00
2025-03-28     101.1282   Call  0.20748     1.00
2025-03-28     111.2411   Call  0.17889     1.00
2025-06-27      91.3564    Put  0.24158     1.00
2025-06-27     101.5071   Call  0.20997     1.00
2025-06-27     111.6578   Call  0.18138     1.00


## The fit

The vanilla rows price by **quadrature** — a fixed clock-by-mixer grid, no draws — so the same
document writes the same factor byte for byte on every run. The solve is Levenberg-Marquardt in
vol points with algorithmic Jacobians, staged: the residual pair, then the leverage, then the
slow pair where the ladder is long enough to see it, then a joint polish.

In [5]:
started = time.time()
factor, report = fit({'Market Prices': {BLOCK: block}}, 'ladder')
print('the fit took %.1f s' % (time.time() - started))
show(report, 'price by QUADRATURE', 'stage 2 (', 'stage 3 (', 'stage 4 (',
     'stage 6 joint', 'evaluations and')

the fit took 15.6 s
LogVar2FJModelPrices.INDEX_A: the vanilla rows price by QUADRATURE, 24 clock nodes x 16 mixer nodes and no draws
stage 2 (alpha, beta): 13 rows, 20 evaluations, residual 7.8430e-03, 5.6s
stage 3 (rho_s, sigma_s): 22 rows, 5 evaluations, residual 8.3306e-03, 2.0s
stage 6 joint polish: 22 rows, 20 evaluations, residual 8.2232e-03, 6.9s
45 evaluations and 41 Jacobians in 14.7s; the inner bootstrap cost 855 pillar passes, each carrying the backward its Newton slope is - 9.9 a sweep over 5 pillars. The seconds go 3.7 on those passes, 7.3 on the Jacobians and 3.7 on everything else


### The forward-variance curve

`xi` is the model's expected forward variance, so it stands beside the market's own squared
at-the-money forward variance and beside the variance-swap strip replicated off each rung's
quotes. The gap to the first is Black's convexity; the gap to the second is the smile's.

In [6]:
show(report, 'market ATM^2')

0.000-0.077y  market ATM^2 20.08% vol, var-swap strip 20.29% vol, fitted xi 20.58% vol (+0.51 / +0.29 vol points)
0.077-0.249y  market ATM^2 20.33% vol, var-swap strip 20.75% vol, fitted xi 21.58% vol (+1.25 / +0.83 vol points)
0.249-0.499y  market ATM^2 20.74% vol, var-swap strip 21.27% vol, fitted xi 22.49% vol (+1.74 / +1.22 vol points)
0.499-0.748y  market ATM^2 21.24% vol, var-swap strip 21.83% vol, fitted xi 23.29% vol (+2.05 / +1.46 vol points)
0.748-0.997y  market ATM^2 21.73% vol, var-swap strip 22.37% vol, fitted xi 24.06% vol (+2.33 / +1.70 vol points)


### What the fit reproduced, per expiry

The at-the-money misses are the inner Newton solve's own tolerance and not a fit residual — the
strip is re-bootstrapped at every iterate, so the term structure is reproduced exactly whatever
the smile does.

In [7]:
show(report, 'vol points over', 'vol points unweighted', 'wing 70-80%', 'convexity 110-120%')

0.077y  RMSE 1.155 vol points over 3 quotes, worst -1.450 at 110% of forward
0.249y  RMSE 0.469 vol points over 3 quotes, worst -0.695 at 110% of forward
0.499y  RMSE 0.216 vol points over 3 quotes, worst -0.339 at 90% of forward
0.748y  RMSE 0.592 vol points over 3 quotes, worst -0.788 at 90% of forward
0.997y  RMSE 0.841 vol points over 3 quotes, worst -1.084 at 90% of forward
RMSE 0.729 vol points unweighted over 15 quotes, 0.729 vega-weighted (the objective's own); the bootstrap's ATM misses +7.3e-11, -4.4e-16, +3.1e-13, +6.6e-12, +3.5e-11
wing 70-80% residual: nothing quoted there
convexity 110-120% residual: +0.893 vol points RMS, worst -1.450


### Identification: which numbers were fitted and which were assumed

This is the half of the report a validator reads first. For every fitted parameter the fit prints
the size of its prior row **as a multiple of one quote row** at the data's own RMS — so a
coordinate whose prior row is several times a quote row is being held, not measured. Beside it
are the stage's singular values and each column's norm in the Jacobian.

In [8]:
show(report, 'identification, ', 'multiple of ONE quote row')

identification, 2 (alpha, beta): singular values 1.095e+00  8.947e-01; column norms Alpha[0y] 2.95e-05, Beta[0y] 6.75e-03
the PRIOR rows there, each as a multiple of ONE quote row at the data's own RMS: Alpha[0y] 7.69x, Beta[0y] 4.4x
identification, 3 (rho_s, sigma_s): singular values 1.373e+00  3.399e-01; column norms Rho_S[0y] 1.87e-02, Sigma_S[0y] 7.64e-03
the PRIOR rows there, each as a multiple of ONE quote row at the data's own RMS: Rho_S[0y] 5.53x, Sigma_S[0y] 3.18x
identification, 6 joint polish: singular values 1.709e+00  9.163e-01  4.819e-01  8.894e-02; column norms Rho_S[0y] 1.19e-02, Beta[0y] 4.73e-03, Sigma_S[0y] 8.41e-03, Alpha[0y] 2.88e-05
the PRIOR rows there, each as a multiple of ONE quote row at the data's own RMS: Rho_S[0y] 8.34x, Beta[0y] 9.2x, Sigma_S[0y] 3.14x, Alpha[0y] 13.6x


A ladder of vanillas alone leaves two directions flat: the residual's tail `Alpha`, and the split
of the spot skew between the residual's skew `Beta` and the leverage product `Rho_S x Sigma_S`.
The report says which rows are carrying them, where each prior came from, and what one standard
error of any prior row costs against one quote missing by one vol point.

In [9]:
show(report, 'leverage prior, TWO rows', 'CLASS PRIOR', 'pinned:', 'ON GUARD',
     'under the floor', 'under the class default = ')

leverage prior, TWO rows - rho_s -0.7000 from the EquityPrice class default; the product rho_s*sigma_s -1.9000 from the EquityPrice class default - at weights 0.02 on rho_s and 0.00833333 on the product, and one standard error of ANY prior row costs 0.00258, which is what one quote missing by one vol point costs on this ladder. The rho_s row also signs the seed and stage 4
LogVar2FJModelPrices.INDEX_A: ON GUARD: c 0.170 at 0y within 0.05 of its C_Min floor 0.12 - the box or the floor is holding theta*, not the data; the factor carries the flag
pinned: not identified by this ladder - Rho_L -0.4000 and Sigma_L 1.0000 are held at the EquityPrice class default, signed by the Rho_S -0.7537 in force at the pin, the longest wing expiry being 0.99726y against the 1.5y stage 4 asks for
the short-dated wings reach the residual and outvote its soft rows where they mean it (its shortest wing expiry is 0.0767123y against the 0.25y Residual_Horizon) - alpha: CLASS PRIOR +44.0000 at spread 0.5, histo

### The guard

This fit carries one. The conditioning share `c = 1 - rho_s^2 - rho_l^2` lands within 0.05 of its
floor `C_Min`, which means the box is holding the answer at that coordinate rather than the data
— the ladder wants more skew than the residual's admissibility leaves room for. The factor
carries the flag, and every mark priced off it carries it too. A guard is not a failure; it is
the fit saying which constraint is binding.

### The stationary law, and the corner

The corner is written from the fitted stationary law, not declared, and the report says what
share of path-days would reach it. A fitted factor whose headroom is zero is one whose
simulation never takes the corner.

In [10]:
show(report, 'stationary log-vol sd', 'residual (alpha, beta)', 'efficiency factor')

residual (alpha, beta) (51.071, -18.152); leverage products rho_s*sigma_s -1.582, rho_l*sigma_l -0.400; c 0.170, conditioning share gamma^2/alpha^2 0.874, c_eff 0.149
efficiency factor (sqrt(c_eff)/0.22)^3 5.37; residual shape alpha*delta_A 0.07671y 1.177, 0.9973y 18.77, against the 0.3 floor (1 strongly non-Gaussian, 15 nearly Gaussian)
stationary log-vol sd 0.573; corner 3.71 with 0.00e+00 of path-days at or above it


### The reserve line

Where the forward smile was not quoted the calibration reports the model half of a reserve
rather than a mark: the sensitivity of the forward skew to `Beta` and to `Rho_S` at the fitted
parameters, and a band in vol points. Notebook 4 composes it with a deal's own derivatives.

In [11]:
show(report, 'reserve line (')

reserve line (0.49863y into 0.49863y, REPORTED - the block is off and these rows are the ladder's own maturities, fitted to nothing), band 0.5 vol points: d(Delta_skew)/dBeta +0.02154 and d(Delta_skew)/dRho_S +3.106 vol points per unit at the 0y bucket - a deal's |dPV/dDelta_skew| x band is utils.LogVar2FJ.skew_reserve of these and its own (dPV/dBeta, dPV/dRho_S)


## The factor the fit wrote

Everything a valuation reads. The four scalars, the residual law, the forward-variance curve, the
four bucketed levers, the corner, and the two reserve fields.

In [12]:
for name in sorted(factor):
    value = factor[name]
    array = getattr(value, 'array', None)
    if array is not None:
        print('%-16s %s' % (name, ', '.join('%g -> %.10g' % (t, v) for t, v in array.tolist())))
    else:
        print('%-16s %s' % (name, value))

Alpha            0 -> 51.0706293
Beta             0 -> -18.15203623
C_Min            0.12
Cap_A            3.709621165520319
Kappa_L          0.5
Kappa_S          6.0
On_Guard         ON GUARD: c 0.170 at 0y within 0.05 of its C_Min floor 0.12
Property_Aliases None
Residual_Law     NIG
Rho_L            -0.4
Rho_S            0 -> -0.8185471327
Sigma_L          1.0
Sigma_S          0 -> 1.933223568
Skew_Gradient    0.0215412133954,3.10631200505
Steps_Per_Year   252.0
Stickiness_Band  0.5
Xi_Curve         0 -> 0.04236290843, 0.0767123 -> 0.04656315032, 0.249315 -> 0.05056607812, 0.49863 -> 0.05424402951, 0.747945 -> 0.057893189


## The same document twice

The vanilla rows price by quadrature and the draws play no part, so a re-run of the identical
document is not "close" — it is the same floats. Every float the factor holds, in hex.

In [13]:
started = time.time()
again, _ = fit({'Market Prices': {BLOCK: block}}, 'ladder_again')
print('the second fit took %.1f s' % (time.time() - started))
first, second = floats_of(factor), floats_of(again)
moved = {k: (v, second.get(k)) for k, v in first.items() if second.get(k) != v}
print('%d floats compared, %d moved' % (len(first), len(moved)))
print(moved or 'byte-identical')

the second fit took 15.0 s
26 floats compared, 0 moved
byte-identical


## What this notebook establishes

- The model is four structural scalars, a forward-variance curve and four bucketed levers, with
  an NIG residual whose drift is no-arbitrage's rather than the fit's.
- The at-the-money term structure is reproduced to the inner solve's tolerance by construction,
  so the objective is the smile.
- The fit states, per parameter, how much of the answer is data and how much is prior. On a
  vanilla-only ladder `Alpha` is the assumed one, and the report says so in two places.
- The corner is a consequence of the fit, not an input, and the fitted factor never reaches it.
- The document is reproducible to the bit.

Notebook 2 measures what would have to be quoted for `Alpha` to be fitted rather than assumed.